In [2]:
import sys
from pathlib import Path

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")

sys.path에 등록된 경로: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast


In [3]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


C:\Users\Hoyoung_Park\PyCharmMiscProject\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def add_yoy_growth(df, value_column='value', group_column='root_hs_code', date_column='date'):
    """
    root_hs_code별로 value 컬럼의 연간 증가율을 계산하여 새로운 컬럼으로 추가합니다.
    """
    df = df.copy()
    # top_company_codes = avg_yoy_by_code.sort_values(by='avg_forecast_yoy', ascending=False).head(company_num)
    df = df.dropna(axis=0)
    df[date_column] = pd.to_datetime(df[date_column])
    df.sort_values(by=[group_column, date_column], inplace=True)

    # YoY (12개월 전 대비 비율 변화율) 계산
    df[f'{value_column}_yoy'] = (
        df.groupby(group_column)[value_column]
        .transform(lambda x: x.pct_change(periods=12))
    )

    return df

In [5]:
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host' : '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

trade_df = fetch_table_data(db_info, 'korea_monthly_trade_data_forecast')

import pandas as pd

# date를 datetime으로 변환
trade_df['date'] = pd.to_datetime(trade_df['date'])

# 결측치 제거
trade_df = trade_df.dropna(subset=['expDlr_forecast_12m'])

# root_hs_code, date로 정렬
trade_df = trade_df.sort_values(['root_hs_code', 'date'])

# 그룹 연산 준비
grouped = trade_df.groupby('root_hs_code')

# 이전 12개월 합계 (trailing)
trade_df['export_trail_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum()
)

# 이후 12개월 합계 (forward)
# shift(-11)은 앞으로 11개월 밀어 rolling 12로 보면 해당 시점 기준 이후 12개월을 의미
trade_df['export_forward_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.shift(-11).rolling(window=12, min_periods=12).sum()
)

# YoY 성장률
trade_df['export_yoy_growth'] = (
    (trade_df['export_forward_12m'] / trade_df['export_trail_12m']) - 1
)

# 필요한 컬럼만 보기
result_df = trade_df[['date', 'root_hs_code',
                      'export_trail_12m', 'export_forward_12m', 'export_yoy_growth']]

# 필요시 최근 데이터만
trade_yoy_growth = result_df[result_df['date'] == '2025-06-30']

# 확인
# print(trade_yoy_growth.head(20))


✅ 'korea_monthly_trade_data_forecast' 테이블에서 235333건의 데이터를 가져왔습니다.


In [6]:
len(trade_yoy_growth['root_hs_code'].unique().tolist())

1021

In [7]:
trade_yoy_growth[trade_yoy_growth['root_hs_code'] == '854232']

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
221343,2025-06-30,854232,7.638940e+10,1.001781e+11,0.311413


In [15]:
company_df = fetch_table_data(db_info, 'korea_company_hscode_map')
company_df = company_df.rename(columns={'hs_code' : 'root_hs_code'})
# company_df.rename(columns={'hs_code_6d': 'root_hs_code'}, inplace=True)

✅ 'korea_company_hscode_map' 테이블에서 762건의 데이터를 가져왔습니다.


In [18]:
company_df

,ticker,Name,root_hs_code
0,A093370,후성,854321
1,A036490,SK머티리얼즈,281290
2,A036490,SK머티리얼즈,8542
3,A104830,원익머트리얼즈,854239
4,A144960,뉴파워프라즈마,854239
...,...,...,...
757,A011784,금호석유,4002590000
758,A011785,금호석유,4002110000
759,A178920,PI첨단소재,3916909000
760,A000070,삼양홀딩스,290723


In [20]:
trade_yoy_growth

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
220472,2025-06-30,1201,7.307999e+04,7.703733e+05,9.541508
220473,2025-06-30,121120,7.812877e+07,7.044364e+07,-0.098365
220474,2025-06-30,1212,5.088124e+08,6.355348e+08,0.249055
220475,2025-06-30,121221,5.081910e+08,6.221920e+08,0.224327
220476,2025-06-30,151550,1.405952e+07,1.253148e+07,-0.108684
...,...,...,...,...,...
221493,2025-06-30,9405,1.652843e+08,1.742183e+08,0.054052
221494,2025-06-30,950300,8.992872e+07,8.993714e+07,0.000094
221495,2025-06-30,961610,6.053126e+06,3.279659e+06,-0.458188
221496,2025-06-30,961900,5.170147e+07,5.344523e+07,0.033727


In [21]:
company_df['root_hs_code'] = company_df['root_hs_code'].astype(str)
trade_yoy_growth['root_hs_code'] = trade_yoy_growth['root_hs_code'].astype(str)

monster_df = pd.merge(company_df, trade_yoy_growth, on='root_hs_code', how='left', indicator=True)

In [22]:
monster_df

,ticker,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
0,A093370,후성,854321,NaT,NaN,NaN,NaN,left_only
1,A036490,SK머티리얼즈,281290,2025-06-30,1.645956e+08,1.770732e+08,0.075808,both
2,A036490,SK머티리얼즈,8542,2025-06-30,1.245768e+11,1.476321e+11,0.185069,both
3,A104830,원익머트리얼즈,854239,2025-06-30,1.145328e+10,1.163399e+10,0.015778,both
4,A144960,뉴파워프라즈마,854239,2025-06-30,1.145328e+10,1.163399e+10,0.015778,both
...,...,...,...,...,...,...,...,...
757,A011784,금호석유,4002590000,2025-06-30,2.590850e+08,2.723613e+08,0.051243,both
758,A011785,금호석유,4002110000,2025-06-30,5.967387e+07,7.255646e+07,0.215883,both
759,A178920,PI첨단소재,3916909000,2025-06-30,2.285735e+07,2.248700e+07,-0.016203,both
760,A000070,삼양홀딩스,290723,2025-06-30,3.184257e+08,3.039058e+08,-0.045599,both


In [12]:
monster_df[monster_df['Code'] == "A004000"]

,hs_code,품목명,Code,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
53,3912399000,셀룰로스 Anycoat,A004000,롯데정밀화학,391239,2025-06-30,368556300.0,404553500.0,0.097671,both


In [9]:
trade_yoy_growth

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
116067,2025-06-30,121120,83255170.0,8.440063e+07,0.013758
116068,2025-06-30,121221,517919500.0,5.963397e+08,0.151414
116069,2025-06-30,151550,12819231.0,8.647523e+06,-0.325426
116070,2025-06-30,151590,5953660.0,6.887209e+06,0.156803
116071,2025-06-30,170199,155429080.0,1.429628e+08,-0.080206
...,...,...,...,...,...
116588,2025-06-30,903180,864757800.0,8.343762e+08,-0.035133
116589,2025-06-30,903190,497127200.0,4.965941e+08,-0.001072
116590,2025-06-30,903289,870932100.0,9.649287e+08,0.107926
116591,2025-06-30,940330,12078255.0,1.106288e+07,-0.084067
